# ESA-NMT: Emotion-Semantic-Aware Neural Machine Translation

**Cross-Family Indian Language Translation: Bengali-Hindi-Telugu**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SSanpui/ESA-NMT/blob/main/ESA_NMT_Research.ipynb)

## Model Overview

ESA-NMT implements a multi-task neural architecture that simultaneously optimizes:
- Translation accuracy using NLLB-200 as base model
- Emotion preservation through XLM-RoBERTa-based classification (8 emotions from Plutchik's wheel)
- Semantic consistency via contrastive learning with cosine similarity

**Achieved Performance:**
- BLEU: 42.66 (Bengali-Hindi), 36.74 (Bengali-Telugu)
- Emotion Classification Accuracy: 77.2% across 8 emotion categories
- Semantic Consistency: 0.92 average cosine similarity
- GPU Memory Reduction: 35% compared to standard training

---

## Prerequisites

**Required Setup:**
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → GPU
2. **Recommended**: V100 or A100 GPU for optimal training speed
3. **Minimum GPU Memory**: 16GB (T4 or higher)

**Training Time Estimates:**
- Full Training (9 epochs): 6-8 hours (V100) / 12-15 hours (T4)
- Quick Demo (1 epoch, 500 samples): 30-45 minutes
- Ablation Studies: 8-10 hours (V100)

---

## Configuration

Configure experiment parameters based on research objectives.

In [ ]:
# Experiment Configuration
RUN_MODE = "full_training"  # Options: "quick_demo", "full_training", "ablation", "weight_tuning"
TRANSLATION_PAIR = "bn-hi"  # Options: "bn-hi", "bn-te"

# Model Architecture
BASE_MODEL = "facebook/nllb-200-distilled-600M"  # NLLB-200 base translation model
EMOTION_MODEL = "xlm-roberta-base"  # Zero-shot cross-lingual emotion detection
SEMANTIC_MODEL = "sentence-transformers/LaBSE"  # Semantic consistency module

# Training Hyperparameters (from paper)
BATCH_SIZE = 1  # Per-device batch size
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 4
NUM_EPOCHS = 9  # Total training epochs
LEARNING_RATE = 2e-5

# Loss Weights for Multi-Objective Learning
ALPHA = 1.0  # Translation loss weight
BETA = 0.3   # Emotion preservation loss weight (experiment with 0.3 and 0.4)
GAMMA = 0.2  # Semantic consistency loss weight (experiment with 0.2 and 0.5)

# Dataset Split (as per paper)
TRAIN_SPLIT = 0.70  # 18,995 samples
VAL_SPLIT = 0.15    # 4,070 samples
TEST_SPLIT = 0.15   # 4,070 samples

print(f"""
{'='*70}
ESA-NMT Configuration
{'='*70}
Mode: {RUN_MODE}
Translation Pair: {TRANSLATION_PAIR}

Architecture:
  Base Translation: {BASE_MODEL}
  Emotion Module: {EMOTION_MODEL} (8 emotions - Plutchik's wheel)
  Semantic Module: {SEMANTIC_MODEL}

Training Setup:
  Batch Size: {BATCH_SIZE} (gradient accumulation: {GRADIENT_ACCUMULATION_STEPS})
  Effective Batch Size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}
  Epochs: {NUM_EPOCHS}
  Learning Rate: {LEARNING_RATE}

Loss Weights:
  α (Translation): {ALPHA}
  β (Emotion): {BETA}
  γ (Semantic): {GAMMA}

Dataset Split:
  Train: {TRAIN_SPLIT*100:.0f}% (18,995 samples)
  Validation: {VAL_SPLIT*100:.0f}% (4,070 samples)
  Test: {TEST_SPLIT*100:.0f}% (4,070 samples)
{'='*70}
""")

## 1. Environment Setup

In [ ]:
# Verify GPU availability and specifications
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU Available: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
    
    if gpu_memory < 15:
        print("\nWARNING: GPU memory may be insufficient for full training.")
        print("Recommended: V100 (16GB) or A100 (40GB)")
        print("Consider using gradient checkpointing or reducing batch size.")
else:
    print("ERROR: No GPU detected!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU")
    raise RuntimeError("GPU is required for training")

## 2. Clone Repository and Setup

In [ ]:
# Clone ESA-NMT repository
!git clone https://github.com/SSanpui/ESA-NMT.git
%cd ESA-NMT

# Verify repository structure
!ls -la

print("\nRepository cloned successfully")

## 3. Install Dependencies

In [ ]:
# Install required packages for ESA-NMT
!pip install -q transformers>=4.30.0 \
    sentence-transformers>=2.2.0 \
    sacrebleu>=2.3.0 \
    rouge-score>=0.1.2 \
    accelerate>=0.20.0 \
    datasets>=2.12.0 \
    torch>=2.0.0 \
    sentencepiece>=0.1.99 \
    protobuf>=3.20.0

# Download NLTK data for evaluation metrics
import nltk
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("All dependencies installed successfully")

## 4. Verify BHT25 Dataset

The BHT25 dataset contains 25,000 parallel literary text samples across Bengali, Hindi, and Telugu, with emotion annotations based on Plutchik's 8-emotion taxonomy.

In [ ]:
import os
import pandas as pd

# Check for BHT25 dataset
if os.path.exists('BHT25_All.csv'):
    df = pd.read_csv('BHT25_All.csv')
    print(f"BHT25 Dataset Found")
    print(f"Total samples: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 3 samples:")
    print(df.head(3))
else:
    print("ERROR: BHT25_All.csv not found")
    print("Please download dataset from: https://huggingface.co/datasets/SSanpui/BHT25")

## 5. Emotion Annotation with XLM-RoBERTa

Zero-shot emotion classification using XLM-RoBERTa-base for cross-lingual emotion detection. The model classifies text into 8 fundamental emotions based on Plutchik's emotion wheel:

1. **Joy** - happiness, contentment, pleasure
2. **Sadness** - sorrow, grief, melancholy
3. **Anger** - fury, resentment, irritation
4. **Fear** - anxiety, apprehension, terror
5. **Surprise** - astonishment, amazement, wonder
6. **Trust** - acceptance, confidence, belief
7. **Disgust** - aversion, revulsion, contempt
8. **Anticipation** - expectation, interest, vigilance

**Note**: This step requires approximately 45-60 minutes for 25,000 samples. Skip if `BHT25_annotated.csv` already exists.

In [ ]:
import os
import pandas as pd

# Check if annotation already exists
if os.path.exists('BHT25_annotated.csv'):
    print("Annotated dataset already exists")
    print("Loading existing annotations...")
    
    df_annotated = pd.read_csv('BHT25_annotated.csv')
    print(f"\nDataset Statistics:")
    print(f"  Total samples: {len(df_annotated)}")
    print(f"  Columns: {df_annotated.columns.tolist()}")
    
    # Display emotion distribution (8 emotions)
    if 'emotion_label' in df_annotated.columns:
        emotion_labels = ['joy', 'sadness', 'anger', 'fear', 'surprise', 'trust', 'disgust', 'anticipation']
        print(f"\n  Emotion Distribution (8 categories - Plutchik's wheel):")
        for i, emotion in enumerate(emotion_labels):
            count = (df_annotated['emotion_label'] == i).sum()
            pct = count / len(df_annotated) * 100
            print(f"    {emotion:15s}: {count:5d} ({pct:5.2f}%)")
    
    # Display semantic similarity statistics
    semantic_cols = [col for col in df_annotated.columns if 'semantic' in col.lower()]
    if semantic_cols:
        print(f"\n  Semantic Similarity Scores:")
        for col in semantic_cols:
            print(f"    {col}: Mean={df_annotated[col].mean():.4f}, Std={df_annotated[col].std():.4f}")

else:
    print("Starting emotion annotation with XLM-RoBERTa...")
    print("Processing 25,000 samples for 8-emotion classification")
    print("Estimated time: 45-60 minutes")
    print("\n" + "="*70)
    
    # Execute annotation script
    !python scripts/annotate_emotions.py \
        --input_file BHT25_All.csv \
        --output_file BHT25_annotated.csv \
        --emotion_model xlm-roberta-base \
        --semantic_model sentence-transformers/LaBSE \
        --num_emotions 8 \
        --batch_size 32
    
    print("\n" + "="*70)
    print("Annotation complete")
    print("Created: BHT25_annotated.csv with 8-emotion labels and semantic scores")

## 6. Progressive Three-Phase Training

ESA-NMT employs a progressive transfer learning strategy:

**Phase 1: Emotion Module Pre-training**
- Train emotion recognition module independently on emotion-annotated data
- Freeze base NLLB-200 translation weights
- Epochs: 3

**Phase 2: Semantic Module Pre-training**
- Train semantic consistency module with contrastive learning
- Freeze base NLLB-200 translation weights
- Epochs: 3

**Phase 3: Joint Multi-Task Fine-tuning**
- Integrate all modules with multi-objective learning
- Fine-tune entire architecture end-to-end
- Combined loss: L_total = α·L_trans + β·L_emotion + γ·L_semantic
- Epochs: 9

This approach prevents catastrophic forgetting while enabling specialized adaptation for literary content.

### 6.1 Quick Demo Mode (Optional)

Test the pipeline with a small subset before full training.

In [ ]:
if RUN_MODE == "quick_demo":
    print("Running Quick Demo")
    print("Training on 500 samples for 1 epoch")
    print(f"Translation pair: {TRANSLATION_PAIR}")
    print("\nEstimated time: 30-45 minutes\n")
    
    !python train_esa_nmt.py \
        --translation_pair {TRANSLATION_PAIR} \
        --base_model {BASE_MODEL} \
        --emotion_model {EMOTION_MODEL} \
        --semantic_model {SEMANTIC_MODEL} \
        --max_samples 500 \
        --num_epochs 1 \
        --batch_size {BATCH_SIZE} \
        --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
        --learning_rate {LEARNING_RATE} \
        --alpha {ALPHA} \
        --beta {BETA} \
        --gamma {GAMMA} \
        --output_dir ./outputs/quick_demo \
        --skip_progressive_training
    
    print("\nQuick demo complete")

### 6.2 Full Progressive Training

In [ ]:
if RUN_MODE == "full_training":
    print("Starting Full Progressive Three-Phase Training")
    print(f"Translation pair: {TRANSLATION_PAIR}")
    print(f"Total samples: 25,000 (Train: 18,995, Val: 4,070, Test: 4,070)")
    print(f"Total epochs: {NUM_EPOCHS}")
    print(f"Steps per epoch: 4,749 (with gradient accumulation)")
    print(f"Total optimization steps: 42,741")
    print("\nEstimated time: 6-8 hours (V100) / 12-15 hours (T4)\n")
    print("="*70)
    
    # Phase 1: Emotion Module Pre-training
    print("\nPhase 1: Emotion Module Pre-training (3 epochs)")
    print("Objective: Train emotion recognition with XLM-RoBERTa")
    print("-"*70)
    
    !python train_esa_nmt.py \
        --translation_pair {TRANSLATION_PAIR} \
        --base_model {BASE_MODEL} \
        --emotion_model {EMOTION_MODEL} \
        --semantic_model {SEMANTIC_MODEL} \
        --train_split {TRAIN_SPLIT} \
        --val_split {VAL_SPLIT} \
        --test_split {TEST_SPLIT} \
        --phase 1 \
        --num_epochs 3 \
        --batch_size {BATCH_SIZE} \
        --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
        --learning_rate {LEARNING_RATE} \
        --freeze_base_model \
        --output_dir ./outputs/phase1_emotion
    
    print("\n" + "="*70)
    print("Phase 1 Complete")
    print("="*70)
    
    # Phase 2: Semantic Module Pre-training
    print("\nPhase 2: Semantic Module Pre-training (3 epochs)")
    print("Objective: Train semantic consistency with contrastive learning")
    print("-"*70)
    
    !python train_esa_nmt.py \
        --translation_pair {TRANSLATION_PAIR} \
        --base_model {BASE_MODEL} \
        --emotion_model {EMOTION_MODEL} \
        --semantic_model {SEMANTIC_MODEL} \
        --train_split {TRAIN_SPLIT} \
        --val_split {VAL_SPLIT} \
        --test_split {TEST_SPLIT} \
        --phase 2 \
        --num_epochs 3 \
        --batch_size {BATCH_SIZE} \
        --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
        --learning_rate {LEARNING_RATE} \
        --freeze_base_model \
        --load_emotion_module ./outputs/phase1_emotion/best_emotion_module.pt \
        --output_dir ./outputs/phase2_semantic
    
    print("\n" + "="*70)
    print("Phase 2 Complete")
    print("="*70)
    
    # Phase 3: Joint Multi-Task Fine-tuning
    print("\nPhase 3: Joint Multi-Task Fine-tuning (9 epochs)")
    print("Objective: Integrate all modules with multi-objective learning")
    print(f"Loss weights: α={ALPHA}, β={BETA}, γ={GAMMA}")
    print("-"*70)
    
    !python train_esa_nmt.py \
        --translation_pair {TRANSLATION_PAIR} \
        --base_model {BASE_MODEL} \
        --emotion_model {EMOTION_MODEL} \
        --semantic_model {SEMANTIC_MODEL} \
        --train_split {TRAIN_SPLIT} \
        --val_split {VAL_SPLIT} \
        --test_split {TEST_SPLIT} \
        --phase 3 \
        --num_epochs {NUM_EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
        --learning_rate {LEARNING_RATE} \
        --alpha {ALPHA} \
        --beta {BETA} \
        --gamma {GAMMA} \
        --load_emotion_module ./outputs/phase1_emotion/best_emotion_module.pt \
        --load_semantic_module ./outputs/phase2_semantic/best_semantic_module.pt \
        --gradient_checkpointing \
        --mixed_precision fp16 \
        --save_steps 500 \
        --output_dir ./outputs/phase3_joint
    
    print("\n" + "="*70)
    print("Phase 3 Complete - Full ESA-NMT Training Finished")
    print("="*70)

### 6.3 Ablation Study

Systematic evaluation of individual module contributions:

1. **Base NLLB**: Translation only (baseline)
2. **+Emotion**: NLLB + Emotion module
3. **+Semantic**: NLLB + Semantic module
4. **Full ESA-NMT**: NLLB + Emotion + Semantic (complete system)

In [ ]:
if RUN_MODE == "ablation":
    print("Running Ablation Studies")
    print(f"Translation pair: {TRANSLATION_PAIR}")
    print("Configurations: Base NLLB, +Emotion, +Semantic, Full ESA-NMT")
    print("\nEstimated time: 8-10 hours (V100)\n")
    print("="*70)
    
    ablation_configs = [
        {
            "name": "base_nllb",
            "use_emotion": False,
            "use_semantic": False,
            "description": "Baseline NLLB-200 (translation only)"
        },
        {
            "name": "emotion_only",
            "use_emotion": True,
            "use_semantic": False,
            "description": "NLLB + Emotion module"
        },
        {
            "name": "semantic_only",
            "use_emotion": False,
            "use_semantic": True,
            "description": "NLLB + Semantic module"
        },
        {
            "name": "full_esa_nmt",
            "use_emotion": True,
            "use_semantic": True,
            "description": "Full ESA-NMT (all modules)"
        }
    ]
    
    for config in ablation_configs:
        print(f"\n{'='*70}")
        print(f"Configuration: {config['name']}")
        print(f"Description: {config['description']}")
        print(f"Emotion Module: {config['use_emotion']}, Semantic Module: {config['use_semantic']}")
        print("-"*70)
        
        !python train_esa_nmt.py \
            --translation_pair {TRANSLATION_PAIR} \
            --base_model {BASE_MODEL} \
            --emotion_model {EMOTION_MODEL} \
            --semantic_model {SEMANTIC_MODEL} \
            --train_split {TRAIN_SPLIT} \
            --val_split {VAL_SPLIT} \
            --test_split {TEST_SPLIT} \
            --use_emotion_module {config['use_emotion']} \
            --use_semantic_module {config['use_semantic']} \
            --num_epochs 5 \
            --batch_size {BATCH_SIZE} \
            --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
            --learning_rate {LEARNING_RATE} \
            --alpha {ALPHA} \
            --beta {BETA} \
            --gamma {GAMMA} \
            --output_dir ./outputs/ablation_{config['name']}
        
        print(f"\nConfiguration {config['name']} complete\n")
    
    print("\n" + "="*70)
    print("All Ablation Studies Complete")
    print("="*70)

### 6.4 Loss Weight Tuning

Experiment with different loss weight combinations:
- Configuration 1: α=1.0, β=0.3, γ=0.2
- Configuration 2: α=1.0, β=0.4, γ=0.5

In [ ]:
if RUN_MODE == "weight_tuning":
    print("Running Loss Weight Tuning Experiments")
    print(f"Translation pair: {TRANSLATION_PAIR}")
    print("Testing two weight configurations")
    print("\nEstimated time: 6-8 hours (V100)\n")
    print("="*70)
    
    weight_configs = [
        {"alpha": 1.0, "beta": 0.3, "gamma": 0.2},
        {"alpha": 1.0, "beta": 0.4, "gamma": 0.5}
    ]
    
    for i, weights in enumerate(weight_configs, 1):
        print(f"\n{'='*70}")
        print(f"Configuration {i}: α={weights['alpha']}, β={weights['beta']}, γ={weights['gamma']}")
        print("-"*70)
        
        !python train_esa_nmt.py \
            --translation_pair {TRANSLATION_PAIR} \
            --base_model {BASE_MODEL} \
            --emotion_model {EMOTION_MODEL} \
            --semantic_model {SEMANTIC_MODEL} \
            --train_split {TRAIN_SPLIT} \
            --val_split {VAL_SPLIT} \
            --test_split {TEST_SPLIT} \
            --num_epochs {NUM_EPOCHS} \
            --batch_size {BATCH_SIZE} \
            --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \
            --learning_rate {LEARNING_RATE} \
            --alpha {weights['alpha']} \
            --beta {weights['beta']} \
            --gamma {weights['gamma']} \
            --output_dir ./outputs/weights_a{weights['alpha']}_b{weights['beta']}_g{weights['gamma']}
        
        print(f"\nWeight configuration {i} complete\n")
    
    print("\n" + "="*70)
    print("Loss Weight Tuning Complete")
    print("="*70)

## 7. Comprehensive Evaluation

Evaluate the trained model on the test set using multiple metrics:

**Translation Quality:**
- BLEU (Bilingual Evaluation Understudy)
- METEOR (Metric for Evaluation of Translation with Explicit ORdering)
- ROUGE-L (Recall-Oriented Understudy for Gisting Evaluation)
- chrF (Character n-gram F-score)

**Emotion Preservation:**
- Classification accuracy across 8 emotion categories
- Per-emotion precision, recall, F1-score
- Confusion matrix analysis

**Semantic Consistency:**
- Cosine similarity between source and translation embeddings
- Cross-lingual semantic alignment score

In [ ]:
print("Running Comprehensive Evaluation")
print(f"Translation pair: {TRANSLATION_PAIR}")
print("Test set: 4,070 samples (15% of BHT25)")
print("\nEvaluating: Translation quality + Emotion preservation + Semantic consistency\n")
print("="*70)

# Determine model directory based on training mode
if RUN_MODE == "full_training":
    model_dir = "./outputs/phase3_joint"
elif RUN_MODE == "ablation":
    model_dir = "./outputs/ablation_full_esa_nmt"
elif RUN_MODE == "quick_demo":
    model_dir = "./outputs/quick_demo"
else:
    model_dir = "./outputs/phase3_joint"

!python evaluate_esa_nmt.py \
    --translation_pair {TRANSLATION_PAIR} \
    --model_dir {model_dir} \
    --base_model {BASE_MODEL} \
    --emotion_model {EMOTION_MODEL} \
    --semantic_model {SEMANTIC_MODEL} \
    --test_split {TEST_SPLIT} \
    --batch_size 8 \
    --compute_all_metrics \
    --save_predictions \
    --output_dir ./outputs/evaluation

print("\n" + "="*70)
print("Evaluation Complete")
print("Results saved to: ./outputs/evaluation")
print("="*70)

## 8. Results Visualization and Analysis

In [ ]:
# Display evaluation metrics
import json
import glob
import os

print("\nEvaluation Results:\n")
print("="*70)

# Load and display main results
results_file = './outputs/evaluation/results.json'
if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print("\nTranslation Quality Metrics:")
    print("-"*70)
    if 'translation_metrics' in results:
        for metric, value in results['translation_metrics'].items():
            print(f"  {metric:20s}: {value:.4f}" if isinstance(value, float) else f"  {metric:20s}: {value}")
    
    print("\nEmotion Preservation Metrics:")
    print("-"*70)
    if 'emotion_metrics' in results:
        for metric, value in results['emotion_metrics'].items():
            if isinstance(value, float):
                print(f"  {metric:20s}: {value:.4f}")
            elif isinstance(value, dict):
                print(f"  {metric}:")
                for k, v in value.items():
                    print(f"    {k:18s}: {v:.4f}" if isinstance(v, float) else f"    {k:18s}: {v}")
    
    print("\nSemantic Consistency Metrics:")
    print("-"*70)
    if 'semantic_metrics' in results:
        for metric, value in results['semantic_metrics'].items():
            print(f"  {metric:20s}: {value:.4f}" if isinstance(value, float) else f"  {metric:20s}: {value}")
else:
    print("Results file not found. Please run evaluation first.")

print("\n" + "="*70)

In [ ]:
# Display visualizations
from IPython.display import Image, display
import glob

print("\nGenerated Visualizations:\n")

viz_files = sorted(glob.glob('./outputs/evaluation/*.png'))
if viz_files:
    for img_file in viz_files:
        print(f"\n{'='*70}")
        print(f"{os.path.basename(img_file)}")
        print("="*70)
        display(Image(filename=img_file, width=800))
else:
    print("No visualizations found. They will be generated during evaluation.")

## 9. Ablation Study Results Comparison

Compare performance across different configurations to validate module contributions.

In [ ]:
if RUN_MODE == "ablation":
    print("\nAblation Study Results Comparison\n")
    print("="*70)
    
    ablation_dirs = [
        ("Base NLLB", "./outputs/ablation_base_nllb"),
        ("+ Emotion", "./outputs/ablation_emotion_only"),
        ("+ Semantic", "./outputs/ablation_semantic_only"),
        ("Full ESA-NMT", "./outputs/ablation_full_esa_nmt")
    ]
    
    import pandas as pd
    
    comparison_data = []
    
    for config_name, output_dir in ablation_dirs:
        results_path = os.path.join(output_dir, 'results.json')
        if os.path.exists(results_path):
            with open(results_path, 'r') as f:
                results = json.load(f)
            
            row = {'Configuration': config_name}
            
            if 'translation_metrics' in results:
                row['BLEU'] = results['translation_metrics'].get('bleu', 0)
                row['METEOR'] = results['translation_metrics'].get('meteor', 0)
                row['ROUGE-L'] = results['translation_metrics'].get('rouge_l', 0)
                row['chrF'] = results['translation_metrics'].get('chrf', 0)
            
            if 'emotion_metrics' in results:
                row['Emotion Acc'] = results['emotion_metrics'].get('accuracy', 0)
            
            if 'semantic_metrics' in results:
                row['Semantic'] = results['semantic_metrics'].get('cosine_similarity', 0)
            
            comparison_data.append(row)
    
    if comparison_data:
        df_comparison = pd.DataFrame(comparison_data)
        print("\n")
        print(df_comparison.to_string(index=False))
        print("\n")
        
        # Save comparison table
        df_comparison.to_csv('./outputs/ablation_comparison.csv', index=False)
        print("Comparison table saved to: ./outputs/ablation_comparison.csv")
    else:
        print("No ablation results found. Please run ablation study first.")
    
    print("\n" + "="*70)

## 10. Download Results Package

In [ ]:
# Package all results for download
print("Packaging results...")

!zip -r esa_nmt_results.zip \
    ./outputs \
    ./checkpoints \
    ./models \
    -x "*.git*" "*__pycache__*" "*.pyc"

print("\nResults packaged successfully")
print("\nPackage contents:")
!unzip -l esa_nmt_results.zip | head -20
print("\nPackage size:")
!ls -lh esa_nmt_results.zip

In [ ]:
# Download results package
from google.colab import files

print("Initiating download...")
print("Note: Large files may take several minutes to download.")

files.download('esa_nmt_results.zip')

print("\nDownload initiated. Check your browser's downloads folder.")

## 11. Summary of Generated Files

In [ ]:
import os

print("\nGenerated Files Summary\n")
print("="*70)

directories = ['./outputs', './checkpoints', './models']

for directory in directories:
    if os.path.exists(directory):
        print(f"\n{directory}:")
        print("-"*70)
        
        total_size = 0
        file_count = 0
        
        for root, dirs, files in os.walk(directory):
            for file in files:
                if not file.startswith('.'):
                    filepath = os.path.join(root, file)
                    size = os.path.getsize(filepath)
                    total_size += size
                    file_count += 1
                    size_mb = size / (1024*1024)
                    rel_path = os.path.relpath(filepath, directory)
                    print(f"  {rel_path:50s} {size_mb:8.2f} MB")
        
        print(f"\n  Total: {file_count} files, {total_size/(1024*1024):.2f} MB")
    else:
        print(f"\n{directory}: Not found")

print("\n" + "="*70)

## Post-Processing and Next Steps

### Completed Tasks
1. Dataset annotation with emotion labels and semantic scores
2. Progressive three-phase training (if full_training mode)
3. Comprehensive evaluation on test set
4. Results visualization and analysis
5. Results package created for download

### Using the Trained Model

**Inference on new text:**
```python
python inference.py \
    --model_dir ./outputs/phase3_joint \
    --source_text "আপনার বাংলা বাক্য এখানে" \
    --translation_pair bn-hi
```

**Deploy to Hugging Face Hub:**
```python
pip install huggingface_hub
huggingface-cli login
python deploy_to_hf.py \
    --model_dir ./outputs/phase3_joint \
    --repo_name ESA-NMT-{TRANSLATION_PAIR} \
    --username YOUR_HF_USERNAME
```

**API Server Deployment:**
```python
python serve_api.py \
    --model_dir ./outputs/phase3_joint \
    --port 8000 \
    --workers 4
```

---

## Expected Performance Benchmarks

Based on reported results in the paper:

**Bengali-Hindi (bn-hi):**
- BLEU: 42.66
- METEOR: 63.04
- ROUGE-L: 0.818
- chrF: 62.60
- Emotion Accuracy: 76.57%
- Semantic Similarity: 0.9290

**Bengali-Telugu (bn-te):**
- BLEU: 36.74
- METEOR: 51.40
- ROUGE-L: 0.904
- chrF: 61.58
- Emotion Accuracy: 77.90%
- Semantic Similarity: 0.9185

**Overall System:**
- 8-Emotion Classification: 77.2% accuracy
- Average Semantic Consistency: 0.92 cosine similarity
- GPU Memory Reduction: 35% vs standard training

---

## Troubleshooting

**Out of Memory (OOM) Errors:**
- Reduce batch size to 1
- Increase gradient accumulation steps
- Enable gradient checkpointing: `--gradient_checkpointing`
- Use mixed precision: `--mixed_precision fp16`

**Slow Training Speed:**
- Upgrade to V100 or A100 GPU (Colab Pro)
- Enable mixed precision training
- Reduce logging frequency
- Use compiled model (PyTorch 2.0+): `--compile_model`

**Low Performance:**
- Verify emotion annotations are correct (not random labels)
- Check that all three training phases completed
- Ensure proper loss weight configuration
- Validate dataset quality and split ratios

**Colab Session Disconnection:**
```javascript
// Run in browser console (F12)
function KeepAlive(){
  console.log("Keep-alive: " + new Date().toTimeString());
  document.querySelector("colab-connect-button").shadowRoot.querySelector("#connect").click();
}
setInterval(KeepAlive, 60000);
```

---

## Citation

If you use ESA-NMT or the BHT25 dataset, please cite:

```bibtex
@article{sanpui2024esanmt,
  title={ESA-NMT: Emotion-Semantic-Aware Neural Machine Translation for Cross-Family Indian Languages},
  author={Sanpui, Sudeshna},
  journal={IEEE Access},
  year={2024},
  note={Accepted for publication}
}
```

**Dataset:**
```bibtex
@dataset{sanpui2024bht25,
  title={BHT25: A Parallel Literary Corpus for Bengali-Hindi-Telugu Translation},
  author={Sanpui, Sudeshna},
  year={2024},
  url={https://huggingface.co/datasets/SSanpui/BHT25}
}
```

---

## Contact and Support

- **Repository**: [github.com/SSanpui/ESA-NMT](https://github.com/SSanpui/ESA-NMT)
- **Dataset**: [huggingface.co/datasets/SSanpui/BHT25](https://huggingface.co/datasets/SSanpui/BHT25)
- **Issues**: Report bugs and feature requests on GitHub Issues

---

**Thank you for using ESA-NMT!**